# S4.2 - Join conditions
**Date completed:** November 2026  
**Status:** completed  
**Key concept:** The condition style decides the key columns in the result and what can be matched

In [0]:
# ============================================================
# S4.2 - Cell 2
# Join Conditions:
# single key, multi-column, expression, non-equi, null-safe
#
# Goal:
# 1. Compare single-key and multi-column joins.
# 2. See list-form vs expression-form join conditions.
# 3. Understand joins when key names differ.
# 4. Practice a non-equi / range join.
# 5. Compare normal equality with null-safe equality.
# 6. Run a real multi-column join on samples.tpch.
# 7. Reconcile predictions with actual results.
#
# Scope:
# Join CONDITIONS only.
#
# Deferred:
# - broadcast joins            -> S4.3
# - physical join strategies   -> S4.4
# - duplicate/cardinality      -> S4.5
# - ambiguous-column cleanup   -> S4.5
# ============================================================

import pyspark.sql.functions as F


print("=== S4.2: JOIN CONDITIONS ===")
print()


# ============================================================
# Step 1 - Controlled Source Data
#
# Price is defined by:
#
# product + region
#
# Therefore:
#
# product alone
# → incomplete business key
#
# product + region
# → complete business key
#
# Order 3:
#
# product = B
# region  = N
#
# There is no matching B/N price.
# ============================================================

orders = spark.createDataFrame(
    [
        (1, "A", "N", 10),
        (2, "A", "S", 5),
        (3, "B", "N", 7),
        (4, "A", "N", 3)
    ],
    """
    order_id INT,
    product STRING,
    region STRING,
    qty INT
    """
)


prices = spark.createDataFrame(
    [
        ("A", "N", 100),
        ("A", "S", 110),
        ("B", "S", 200)
    ],
    """
    product STRING,
    region STRING,
    price INT
    """
)


print("=== STEP 1: ORDERS ===")
orders.orderBy("order_id").show()


print("=== STEP 1: PRICES ===")
prices.orderBy("product", "region").show()


In [0]:
# ============================================================
# Step 2 - Single-Key Join
#
# Intentionally use only:
#
# product
#
# This is an incomplete key for the business relationship.
#
# Product A has two price rows:
#
# A/N
# A/S
#
# Orders 1, 2 and 4 are product A.
#
# Each of those orders matches TWO price rows.
#
# Order 3 is product B and matches B/S once.
#
# Expected:
#
# 2 + 2 + 1 + 2 = 7 rows
#
# This demonstrates why choosing the correct join key matters.
# Detailed fan-out/cardinality analysis is covered in S4.5.
# ============================================================

print("=== STEP 2: SINGLE KEY - product only ===")

one_key = (orders.join(prices,on="product", how="left"))

one_key_count = (one_key.count())

print(f"Predicted rows: 7")

print(f"Actual rows   : {one_key_count}")

print(f"Match         : {one_key_count == 7}")

one_key.orderBy("order_id",orders["region"]).show()

In [0]:
# ============================================================
# Step 3 - Multi-Column Join: List Form
#
# Correct business key:
#
# product + region
#
# Because both DataFrames use the SAME key names, we can use:
#
# on=["product", "region"]
#
# Spark treats these columns as common join keys.
#
# The result contains only one:
#
# product
# region
#
# column pair.
#
# Expected:
#
# one output row per order
# → 4 rows
#
# Order 3 has no B/N price:
#
# price = NULL
# ============================================================

print(
    "=== STEP 3: MULTI-COLUMN JOIN - LIST FORM ==="
)


multi_list = (orders.join(prices,on=["product","region"],how="left"))

multi_list_count = (multi_list.count())

print(f"Predicted rows: 4")

print(f"Actual rows   : {multi_list_count}")

print(f"Match         : {multi_list_count == 4}")

print(f"Columns       : {multi_list.columns}")

multi_list.orderBy("order_id").show()

In [0]:
# ============================================================
# Step 4 - Multi-Column Join: Expression Form
#
# Same logical condition as Step 3:
#
# orders.product = prices.product
#
# AND
#
# orders.region = prices.region
#
#
# Expression form is useful when:
#
# - key names differ
# - conditions are more complex
# - non-equality predicates are required
#
#
# Important:
#
# With an explicit expression Spark can retain the join
# columns from both DataFrames.
#
# Detailed ambiguous-column handling is covered in S4.5.
# ============================================================

print("=== STEP 4: MULTI-COLUMN JOIN - EXPRESSION FORM ===")

o = (orders.alias("o"))

p = (prices.alias("p"))

multi_expression = (o.join(p,
        (F.col("o.product") == F.col("p.product")) &
        (F.col("o.region") == F.col("p.region")),"left"))

multi_expression_count = (multi_expression.count())

print(f"Predicted rows: 4")

print(f"Actual rows   : {multi_expression_count}")

print(f"Match         : {multi_expression_count == 4}")

print(f"Columns       : {multi_expression.columns}")

multi_expression.show()

In [0]:
# ============================================================
# Step 5 - Different Column Names
#
# Rename the right-side keys:
#
# product -> p_product
# region  -> p_region
#
#
# A list-form join such as:
#
# on=["product", "region"]
#
# requires those column names to exist on BOTH DataFrames.
#
#
# When the names differ, an explicit expression is a natural
# way to define the relationship.
# ============================================================

print("=== STEP 5: DIFFERENT COLUMN NAMES ===")

price_alt = (
    prices

    .withColumnRenamed("product","p_product")

    .withColumnRenamed("region", "p_region"))

oa = (orders.alias("o"))

pa = (price_alt.alias("p"))

different_names = (
    oa
    .join(pa,
        (F.col("o.product")==F.col("p.p_product")) & (F.col("o.region") == F.col("p.p_region")),"left"))

different_names_count = (different_names.count())

print(f"Predicted rows: 4")

print(f"Actual rows   : {different_names_count}")

print(f"Match         : {different_names_count == 4}")

print(f"Columns       : {different_names.columns}")

different_names.show()

In [0]:
# ============================================================
# Step 6 - Non-Equi / Range Join
#
# A join condition does not have to use only:
#
# ==
#
# Here each order quantity is matched to a band:
#
# small  : 0 - 5
# medium : 6 - 9
# large  : 10 - 99
#
#
# Condition:
#
# qty >= lo
#
# AND
#
# qty <= hi
#
#
# The bands in this controlled example do not overlap,
# therefore each order is expected to match one band.
#
# Expected:
#
# 4 rows
# ============================================================

print("=== STEP 6: NON-EQUI / RANGE JOIN ===")


bands = spark.createDataFrame(
    [
        ("small", 0, 5),
        ("medium", 6, 9),
        ("large", 10, 99)
    ],
    """
    band STRING,
    lo INT,
    hi INT
    """
)


banded = (
    orders.join(bands,
        (orders["qty"] >= bands["lo"]) &
        (orders["qty"] <= bands["hi"]), "left"))

banded_count = (banded.count())

print(f"Predicted rows: 4")

print(f"Actual rows   : {banded_count}")

print(f"Match         : {banded_count == 4}")


(
    banded
    .select(
        "order_id",
        "qty",
        "band")
    .orderBy("order_id").show())

In [0]:
# ============================================================
# Step 7 - NULL Join-Key Behaviour
#
# Normal equality:
#
# NULL == NULL
#
# does NOT evaluate TRUE.
#
# Therefore two NULL join keys do not match under normal
# equality.
#
#
# Spark also provides null-safe equality:
#
# eqNullSafe()
#
# SQL equivalent:
#
# <=>
#
#
# NULL <=> NULL
#
# evaluates TRUE.
# ============================================================

print("=== STEP 7: NULL JOIN CONDITIONS ===")

left_null = spark.createDataFrame(
    [
        (1, "L1"),
        (None, "L_NULL")
    ],
    """
    id INT,
    left_value STRING
    """
)

right_null = spark.createDataFrame(
    [
        (1, "R1"),
        (None, "R_NULL")
    ],
    """
    id INT,
    right_value STRING
    """)

ln = (left_null.alias("l"))

rn = (right_null.alias("r"))

# ------------------------------------------------------------
# Normal equality
#
# Expected:
#
# id 1 matches
# NULL does not match NULL
#
# → 1 row
# ------------------------------------------------------------

normal_null_join = (ln .join(rn,F.col("l.id") ==  F.col("r.id"),"inner"))

normal_null_count = (normal_null_join.count())

print(f"Normal equality predicted: 1")

print(f"Normal equality actual   : "f"{normal_null_count}")

print(f"Match                    : "f"{normal_null_count == 1}")

# ------------------------------------------------------------
# Null-safe equality
#
# Expected:
#
# id 1 matches
# NULL matches NULL
#
# → 2 rows
# ------------------------------------------------------------

null_safe_join = (ln.join(rn,F.col("l.id").eqNullSafe(F.col("r.id")),"inner"))

null_safe_count = (null_safe_join.count())

print(f"Null-safe predicted      : 2")

print(f"Null-safe actual         : "f"{null_safe_count}")

print(f"Match                    : "f"{null_safe_count == 2}")

(
    null_safe_join

    .select(
        F.col("l.id").alias("left_id"),
        F.col("r.id").alias("right_id"),
        F.col("l.left_value"),
        F.col("r.right_value")).show()
)

In [0]:
# ============================================================
# Step 8 - Controlled Validation
#
# Assertions are appropriate here because the tiny source
# data is fully controlled and the expected answers are known.
# ============================================================

print(
    "=== STEP 8: CONTROLLED VALIDATION ==="
)


assert (one_key_count == 7
), "Single-key prediction failed"


assert (multi_list_count == 4
), "Multi-column list join prediction failed"

assert (multi_expression_count == 4
), "Expression join prediction failed"

assert (different_names_count == 4
), "Different-name join prediction failed"

assert (banded_count == 4
), "Range join prediction failed"

assert (normal_null_count ==1
), "Normal NULL equality prediction failed"

assert (null_safe_count == 2
), "NULL-safe equality prediction failed"

print("All controlled S4.2 predictions matched.")

print()

In [0]:
# ============================================================
# Step 9 - Real Data: TPC-H Multi-Column Join
#
# Real relationship:
#
# lineitem
#      ↓
# partsupp
#
#
# lineitem keys:
#
# l_partkey
# l_suppkey
#
#
# partsupp keys:
#
# ps_partkey
# ps_suppkey
#
#
# The relationship therefore requires BOTH:
#
# partkey
# +
# suppkey
#
#
# This gives us a genuine real-data multi-column join.
# ============================================================

print(
    "=== STEP 9: TPC-H MULTI-COLUMN JOIN ==="
)


li = (
    spark.table("samples.tpch.lineitem")

    .select(
        "l_orderkey",
        "l_linenumber",
        "l_partkey",
        "l_suppkey"))

ps = (
    spark.table("samples.tpch.partsupp")

    .select(
        "ps_partkey",
        "ps_suppkey",
        "ps_supplycost"))

li_alias = (li.alias("li"))

ps_alias = (ps.alias("ps"))

tpch_condition = (
    (F.col("li.l_partkey") ==  F.col("ps.ps_partkey")) &
    (F.col("li.l_suppkey") == F.col("ps.ps_suppkey")))

n_lineitem = (li.count())

n_partsupp = (ps.count())

n_tpch_join = (li_alias.join(ps_alias, tpch_condition,"inner").count())

# ------------------------------------------------------------
# LEFT ANTI gives lineitems for which no matching
# (partkey, suppkey) exists in partsupp.
# ------------------------------------------------------------

n_tpch_anti = (
    li_alias
    .join(ps_alias,tpch_condition,"left_anti").count())

print(f"lineitem rows          : "f"{n_lineitem:,}")

print(f"partsupp rows       : "f"{n_partsupp:,}")

print(f"multi-column join rows : "f"{n_tpch_join:,}")

print(f"unmatched lineitems    : "f"{n_tpch_anti:,}")

print(f"join == lineitem rows  : "f"{n_tpch_join == n_lineitem}")

print(f"left_anti == 0         : "f"{n_tpch_anti == 0}")


In [0]:
# ============================================================
# Step 10 - Show Real Joined Rows
#
# Select explicitly because the two tables use differently
# named key columns.
# ============================================================

print("=== STEP 10: REAL JOIN SAMPLE ===")

(
    li_alias

    .join(
        ps_alias,
        tpch_condition,
        "inner"
    )

    .select(
        F.col("li.l_orderkey"),
        F.col("li.l_linenumber"),
        F.col("li.l_partkey"),
        F.col("li.l_suppkey"),
        F.col("ps.ps_supplycost"))

    .orderBy("l_orderkey","l_linenumber")

    .show(10,truncate=False))

In [0]:
# ============================================================
# Step 11 - Final Checkpoint Output
#
# These values should be copied into the final S4.2 %md note.
# ============================================================

print("=== STEP 11: S4.2 CHECKPOINT ===")


print(
    f"Single-key rows                  : "
    f"{one_key_count}")

print(
    f"Multi-column list rows           : "
    f"{multi_list_count}")

print(
    f"Multi-column expression rows     : "
    f"{multi_expression_count}")

print(
    f"Different-name key rows          : "
    f"{different_names_count}")

print(
    f"Range join rows                  : "
    f"{banded_count}")

print(
    f"Normal NULL equality rows        : "
    f"{normal_null_count}")

print(
    f"NULL-safe equality rows          : "
    f"{null_safe_count}")

print(
    f"TPC-H lineitem rows              : "
    f"{n_lineitem:,}")

print(
    f"TPC-H partsupp rows              : "
    f"{n_partsupp:,}")

print(
    f"TPC-H multi-column join rows     : "
    f"{n_tpch_join:,}")

print(
    f"TPC-H unmatched lineitems        : "
    f"{n_tpch_anti:,}")


In [0]:
# ============================================================
# Step 12 - Final Learning Summary
# ============================================================

print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
SINGLE KEY

on="product"

→ equi-join using one common key
→ may be wrong if the business relationship needs more keys


MULTI-COLUMN LIST FORM

on=["product", "region"]

→ requires those names on both DataFrames
→ equi-join
→ common key columns appear once


MULTI-COLUMN EXPRESSION

(left.a == right.a) &
(left.b == right.b)

→ flexible condition
→ useful for complex joins
→ both sides' key columns can remain


DIFFERENT KEY NAMES

left.product == right.p_product

→ expression form is a natural approach


NON-EQUI JOIN

qty >= lo
AND
qty <= hi

→ join conditions are not limited to equality


NORMAL NULL EQUALITY

NULL == NULL
→ not TRUE
→ no match


NULL-SAFE EQUALITY

eqNullSafe()

SQL:
<=>

NULL <=> NULL
→ TRUE
→ match


REAL MULTI-COLUMN JOIN

lineitem
→ l_partkey + l_suppkey

partsupp
→ ps_partkey + ps_suppkey


CORE RULE

Choose the join condition from the actual
business relationship, not simply from the
columns that happen to have similar names.
""")


print(
    "=== S4.2 CELL 2 COMPLETED ==="
)

# S4.2 — Key Takeaways: Join Conditions

## 1. Join Condition Styles

| Style | Example | Join-key columns in result |
|---|---|---|
| Single name | `on="product"` | common key appears once |
| List of names | `on=["product", "region"]` | common keys appear once |
| Expression | `(a.x == b.y) & (a.p == b.q)` | columns from both sides can remain |

For string or list-form joins, the named columns must exist on both DataFrames.

When key names differ, expression form is a natural approach:

```python
left["product"] == right["p_product"]
```

Another option is to rename the columns first and then use list form.

---

## 2. Use the Full Business Key

The join condition should represent the real business relationship.

In our example, price depends on:

```text
product + region
```

Joining only on:

```text
product
```

was incomplete.

Observed:

```text
4 source orders
      ↓
join on product only
      ↓
7 rows
```

Why?

Product `A` has two price rows:

```text
A / N
A / S
```

so several orders matched both.

Using the full key:

```python
on=["product", "region"]
```

returned:

```text
4 rows
```

### Core rule

```text
Choose the join condition from the business relationship,
not simply from columns that look similar.
```

---

## 3. Duplicate Column Names After an Incomplete Join

When we joined only on:

```python
on="product"
```

`product` became the common join key and appeared once.

But both DataFrames also contained:

```text
region
```

and `region` was not part of the join condition.

Therefore the result contained:

```text
orders.region
prices.region
```

both named `region`.

This made:

```python
orderBy("region")
```

ambiguous.

We could explicitly reference:

```python
orders["region"]
```

or give the columns clearer names such as:

```text
order_region
price_region
```

Detailed ambiguous-column handling is covered in **S4.5**.

---

## 4. Missing Match

Order 3 contains:

```text
product = B
region  = N
```

There is no matching `B/N` price row.

With a left join:

```text
order remains
price = NULL
```

Therefore a left join can expose missing reference data without removing the left-side row.

---

## 5. Non-Equi Joins

Join conditions are not limited to equality.

Example:

```python
orders["qty"] >= bands["lo"]
```

and:

```python
orders["qty"] <= bands["hi"]
```

This allows range-based matching.

Our quantity bands were:

```text
small  = 0–5
medium = 6–9
large  = 10–99
```

Observed by `order_id`:

```text
order 1 → qty 10 → large
order 2 → qty 5  → small
order 3 → qty 7  → medium
order 4 → qty 3  → small
```

### Important

If ranges overlap, one source row can match multiple range rows.

Therefore range definitions must be validated.

---

## 6. NULL Join Keys

Normal equality does not match two NULL keys.

```text
NULL == NULL
→ not TRUE
→ no match
```

Spark provides null-safe equality:

```python
eqNullSafe()
```

SQL equivalent:

```sql
<=>
```

With null-safe equality:

```text
NULL <=> NULL
→ TRUE
→ match
```

Observed:

```text
Normal equality rows = 1
Null-safe rows       = 2
```

Use null-safe equality only when matching NULL values is part of the business rule.

---

## 7. Reconciliation

Join counts should be checked after execution.

For a left join:

```text
left join rows == left source rows
```

means no left row produced more than one joined row.

If:

```text
left join rows > left source rows
```

then at least one left row matched multiple right-side rows.

This does **not** prove that the right-side key is globally unique. It only tells us what happened for the left rows participating in this join.

Another useful check is:

```text
inner join rows == left rows
AND
left_anti rows == 0
```

If both are true, every left row matched exactly one right-side row for that join.

Detailed cardinality and duplicate-key behaviour is covered in **S4.5**.

---

## 8. Observed on Controlled Data

All controlled predictions matched.

```text
Single-key join on product
→ 7 rows

Multi-column list join
→ 4 rows

Multi-column expression join
→ 4 rows

Different-name expression join
→ 4 rows

Range join
→ 4 rows

Normal NULL equality
→ 1 row

NULL-safe equality
→ 2 rows
```

For list form:

```python
on=["product", "region"]
```

result columns were:

```text
product
region
order_id
qty
price
```

The common join keys appeared once.

For expression form, both sides' `product` and `region` columns remained available.

---

## 9. Observed on samples.tpch

Real multi-column relationship:

```text
lineitem
→ l_partkey + l_suppkey

partsupp
→ ps_partkey + ps_suppkey
```

Observed:

```text
lineitem rows             = 29,999,795
partsupp rows             = 4,000,000
multi-column join rows    = 29,999,795
unmatched lineitems       = 0

join == lineitem rows     = True
```

Because:

```text
unmatched lineitems = 0
```

and:

```text
joined rows = lineitem rows
```

every lineitem matched exactly one `partsupp` row in this dataset.

---

## 10. Checkpoint

### Q1. Why did joining only on `product` produce 7 rows?

Because `product` was not the complete business key. Product `A` had two matching price rows, so several orders matched more than once.

---

### Q2. Why did `(product, region)` return 4 rows?

Because that pair represented the complete key for the controlled price relationship.

---

### Q3. What is the difference between list form and expression form?

List form:

```python
on=["product", "region"]
```

uses same-named equality keys and keeps the common join keys once.

Expression form:

```python
(left.product == right.product) &
(left.region == right.region)
```

is more flexible and can retain columns from both sides.

---

### Q4. When would you use expression form?

When:

```text
column names differ
multiple conditions are required
non-equality logic is required
null-safe equality is required
```

---

### Q5. Does `NULL == NULL` match?

No.

Normal equality does not treat two NULL values as equal.

Use:

```python
eqNullSafe()
```

when NULL-to-NULL matching is intentionally required.

---

### Q6. What did the TPC-H multi-column checkpoint prove?

For the tested `lineitem → partsupp` relationship:

```text
all lineitems matched
and
each lineitem produced exactly one joined row
```

using:

```text
partkey + suppkey
```

---

## 11. Core Understanding

```text
One same-name key?
→ on="key"

Multiple same-name keys?
→ on=["key1", "key2"]

Different key names?
→ expression condition

Complex AND conditions?
→ expression condition

Range relationship?
→ non-equi condition

NULL should match NULL?
→ eqNullSafe()

Unexpected extra rows?
→ inspect whether the join condition is incomplete
→ deeper cardinality analysis in S4.5
```

**S4.2 — Join Conditions COMPLETED.**